# 01 — Environment check

One-off verification that the COMP9517 DL pipeline venv can see the **RTX 2050** and that the packages we need are installed.

**Kernel to select in Cursor:** `COMP9517-DL (RTX2050)` (`comp9517-dl`)

If `torch.cuda.is_available()` is `False`, stop here and fix the install before any training notebooks.

## 1. Project constants

Seed and data path are fixed for the whole pipeline so Nate and Abdoali stay reproducible across notebooks.

In [1]:
from pathlib import Path

SEED = 42

# Native-resolution iNat subset (train / val / test), 500 classes
DATA_ROOT = Path(r"C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset")

# Pipeline root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().resolve().parent
if PROJECT_ROOT.name != "dl_pipeline":
    # Fallback if the notebook cwd is already the pipeline root
    PROJECT_ROOT = Path(r"C:\Users\Abdoali\Comp9517\Group_project\dl_pipeline")

CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR = PROJECT_ROOT / "src"

print(f"SEED          = {SEED}")
print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"DATA_ROOT     = {DATA_ROOT}  exists={DATA_ROOT.is_dir()}")
print(f"checkpoints   = {CHECKPOINTS_DIR.is_dir()}, results = {RESULTS_DIR.is_dir()}, src = {SRC_DIR.is_dir()}")
for split in ("train", "val", "test"):
    p = DATA_ROOT / split
    n_classes = len([d for d in p.iterdir() if d.is_dir()]) if p.is_dir() else 0
    print(f"  {split}: {p}  class_folders={n_classes}")

SEED          = 42
PROJECT_ROOT  = C:\Users\Abdoali\Comp9517\Group_project\dl_pipeline
DATA_ROOT     = C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset  exists=True
checkpoints   = True, results = True, src = True
  train: C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset\train  class_folders=500
  val: C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset\val  class_folders=500
  test: C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset\test  class_folders=500


## 2. Package versions

Record these in the report's experimental-setup section.

In [2]:
import sys
import platform

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns
import tqdm
import torch
import torchvision

print(f"Python        : {sys.version.split()[0]}  ({platform.platform()})")
print(f"torch         : {torch.__version__}")
print(f"torchvision   : {torchvision.__version__}")
print(f"numpy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print(f"matplotlib    : {matplotlib.__version__}")
print(f"seaborn       : {sns.__version__}")
print(f"tqdm          : {tqdm.__version__}")

Python        : 3.12.10  (Windows-11-10.0.26200-SP0)
torch         : 2.13.0+cu126
torchvision   : 0.28.0+cu126
numpy         : 2.4.4
pandas        : 3.0.5
scikit-learn  : 1.9.0
matplotlib    : 3.11.1
seaborn       : 0.13.2
tqdm          : 4.70.0


## 3. GPU / CUDA visibility

Driver reports a *maximum* CUDA toolkit it can run; PyTorch ships its own CUDA runtime (here **12.6**). A newer driver (ours: 13.2) can run an older runtime — that is expected and correct.

**4GB VRAM reminder:** later training should use mixed precision (`torch.cuda.amp`) and modest batches (16–32 @ 224×224 for ResNet-18).

In [3]:
print(f"cuda.is_available() : {torch.cuda.is_available()}")
print(f"torch.version.cuda  : {torch.version.cuda}")
print(f"cuDNN enabled       : {torch.backends.cudnn.enabled}")

assert torch.cuda.is_available(), (
    "CUDA not visible. Reinstall with:\n"
    "  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126\n"
    "and select kernel COMP9517-DL (RTX2050) in Cursor."
)

idx = 0
props = torch.cuda.get_device_properties(idx)
total_gb = props.total_memory / (1024 ** 3)
print(f"device name         : {torch.cuda.get_device_name(idx)}")
print(f"compute capability  : {torch.cuda.get_device_capability(idx)}")  # expect (8, 6) Ampere
print(f"total VRAM          : {total_gb:.2f} GiB")
print(f"multi-processor cnt : {props.multi_processor_count}")

cuda.is_available() : True
torch.version.cuda  : 12.6
cuDNN enabled       : True
device name         : NVIDIA GeForce RTX 2050
compute capability  : (8, 6)
total VRAM          : 4.00 GiB
multi-processor cnt : 16


## 4. Tiny CUDA smoke test

Allocates a small matrix on GPU and multiplies it. Safe for 4GB — uses only a few MB.

In [4]:
device = torch.device("cuda")
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

a = torch.randn(512, 512, device=device)
b = torch.randn(512, 512, device=device)
c = a @ b
torch.cuda.synchronize()

alloc_mb = torch.cuda.memory_allocated(device) / (1024 ** 2)
print(f"matmul result sum   : {c.sum().item():.4f}")
print(f"memory allocated    : {alloc_mb:.1f} MiB")
print("CUDA smoke test OK")

del a, b, c
torch.cuda.empty_cache()

matmul result sum   : -14077.7871
memory allocated    : 11.1 MiB
CUDA smoke test OK


## 5. Mixed-precision sanity check

We will use AMP by default in training notebooks to stretch 4GB VRAM. This cell only confirms autocast runs.

In [5]:
from torch.amp import autocast

x = torch.randn(32, 3, 64, 64, device=device)
with autocast("cuda", dtype=torch.float16):
    y = torch.nn.functional.conv2d(x, torch.randn(8, 3, 3, 3, device=device))
print(f"AMP output dtype    : {y.dtype}")  # expect torch.float16
print("AMP smoke test OK")
del x, y
torch.cuda.empty_cache()

AMP output dtype    : torch.float16
AMP smoke test OK


## 6. Done

If every cell above ran without assertion errors:

1. Kernel `COMP9517-DL (RTX2050)` is wired correctly in Cursor.
2. CUDA + AMP work on the RTX 2050.
3. Data path and folder layout are visible.

**Next notebook:** `02_data_pipeline.ipynb` (dataset / dataloaders + visual sanity check).